# 01 — Exploratory Data Analysis (EDA)
## Telco Customer Churn

### Objetivo
Realizar uma análise exploratória completa da base de clientes para:
- avaliar qualidade e consistência dos dados;
- entender a distribuição de `Churn`;
- investigar variáveis numéricas e categóricas;
- identificar segmentos com maior churn;
- levantar hipóteses para feature engineering;
- gerar insights de negócio que orientem a modelagem.

> **Importante:** a EDA é descritiva. Associação com churn não implica causalidade.


## 1. Imports e configuração


In [ ]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

SRC_PATH = Path("../src").resolve()
if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

from data_preprocessing import load_telco_data

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")


## 2. Carregamento dos dados

O arquivo bruto deve estar em `data/raw/telco_customer_churn.xlsx`.


In [ ]:
DATA_PATH = Path("../data/raw/telco_customer_churn.xlsx")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Arquivo não encontrado: {DATA_PATH.resolve()}"
    )

df = load_telco_data(DATA_PATH)

print(f"Arquivo: {DATA_PATH.name}")
print(f"Dimensão: {df.shape[0]:,} linhas x {df.shape[1]} colunas")
df.head()


## 3. Visão geral e qualidade dos dados


In [ ]:
df.info()


In [ ]:
quality_summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "n_unique": df.nunique(dropna=False),
    "missing": df.isna().sum(),
    "missing_pct": df.isna().mean() * 100,
})
quality_summary.sort_values(["missing", "n_unique"], ascending=[False, True])


In [ ]:
print(f"Registros duplicados: {df.duplicated().sum():,}")
print(f"Clientes únicos: {df['customerID'].nunique():,}")
print(f"Linhas totais: {len(df):,}")
print(f"Valores ausentes totais: {df.isna().sum().sum():,}")
print("customerID único por linha:", df['customerID'].nunique() == len(df))


### Interpretação
- `customerID` deve ser usado apenas como identificador e excluído da modelagem.
- Duplicidades e missing values devem ser documentados antes de qualquer tratamento.
- No seu arquivo, `TotalCharges` já aparece numérico, mas o pipeline continua preparado para conversão segura.


## 4. Distribuição da variável alvo


In [ ]:
target_counts = (
    df["Churn"].value_counts().sort_index()
    .rename(index={0: "Não Churn", 1: "Churn"})
    .to_frame("clientes")
)
target_counts["percentual"] = target_counts["clientes"] / target_counts["clientes"].sum() * 100
target_counts


In [ ]:
churn_rate = df["Churn"].mean()
print(f"Taxa de churn: {churn_rate:.2%}")
print(f"Taxa de permanência: {1-churn_rate:.2%}")


In [ ]:
ax = df["Churn"].value_counts().sort_index().plot(
    kind="bar", figsize=(7,4), title="Distribuição da variável alvo"
)
ax.set_xlabel("Classe")
ax.set_ylabel("Número de clientes")
ax.set_xticklabels(["Não Churn", "Churn"], rotation=0)
plt.tight_layout()
plt.show()


### Leitura
O desbalanceamento é moderado. Portanto, não devemos aplicar SMOTE automaticamente. Na modelagem, vale comparar modelos com e sem ponderação de classes e usar ROC-AUC, PR-AUC, Recall e F1 além de Accuracy.


## 5. Separação conceitual das variáveis


In [ ]:
identifier_cols = ["customerID"]
target_col = "Churn"
numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
categorical_cols = [c for c in df.columns if c not in identifier_cols + [target_col] + numeric_cols]

print("Numéricas:", numeric_cols)
print("Categóricas:", categorical_cols)


> `SeniorCitizen` é armazenada como 0/1, mas conceitualmente é uma variável categórica binária.


## 6. Estatísticas descritivas das variáveis numéricas


In [ ]:
df[numeric_cols].describe().T


In [ ]:
df.groupby("Churn")[numeric_cols].agg(["mean", "median", "std"]).round(2)


### O que observar
- `tenure`: churners ficam menos tempo?
- `MonthlyCharges`: churners pagam mais por mês?
- `TotalCharges`: diferenças podem refletir tempo de relacionamento e valor acumulado.


## 7. Distribuições numéricas por Churn


In [ ]:
for col in numeric_cols:
    fig, ax = plt.subplots(figsize=(8,4))
    for churn_value, label in [(0,"Não Churn"),(1,"Churn")]:
        df.loc[df["Churn"] == churn_value, col].plot(
            kind="hist", bins=30, alpha=0.5, density=True, ax=ax, label=label
        )
    ax.set_title(f"Distribuição de {col} por Churn")
    ax.set_xlabel(col)
    ax.set_ylabel("Densidade")
    ax.legend()
    plt.tight_layout()
    plt.show()


## 8. Boxplots das variáveis numéricas


In [ ]:
for col in numeric_cols:
    fig, ax = plt.subplots(figsize=(7,4))
    data_0 = df.loc[df["Churn"] == 0, col].dropna()
    data_1 = df.loc[df["Churn"] == 1, col].dropna()
    ax.boxplot([data_0, data_1], tick_labels=["Não Churn", "Churn"])
    ax.set_title(f"{col} por Churn")
    ax.set_ylabel(col)
    plt.tight_layout()
    plt.show()


## 9. Função auxiliar para análise categórica


In [ ]:
def churn_by_category(data: pd.DataFrame, column: str) -> pd.DataFrame:
    result = (
        data.groupby(column, observed=False)
        .agg(clientes=("Churn","size"), churns=("Churn","sum"), churn_rate=("Churn","mean"))
        .reset_index()
    )
    result["churn_rate_pct"] = result["churn_rate"] * 100
    return result.sort_values(["churn_rate", "clientes"], ascending=[False, False]).reset_index(drop=True)


## Contract — churn por categoria


In [ ]:
result = churn_by_category(df, "Contract")
result


In [ ]:
plot_data = result.sort_values("churn_rate_pct", ascending=True)
ax = plot_data.plot(
    x="Contract", y="churn_rate_pct", kind="barh", figsize=(8,4),
    legend=False, title="Taxa de Churn por Contract"
)
ax.set_xlabel("Churn (%)")
ax.set_ylabel("Contract")
plt.tight_layout()
plt.show()


## InternetService — churn por categoria


In [ ]:
result = churn_by_category(df, "InternetService")
result


In [ ]:
plot_data = result.sort_values("churn_rate_pct", ascending=True)
ax = plot_data.plot(
    x="InternetService", y="churn_rate_pct", kind="barh", figsize=(8,4),
    legend=False, title="Taxa de Churn por InternetService"
)
ax.set_xlabel("Churn (%)")
ax.set_ylabel("InternetService")
plt.tight_layout()
plt.show()


## PaymentMethod — churn por categoria


In [ ]:
result = churn_by_category(df, "PaymentMethod")
result


In [ ]:
plot_data = result.sort_values("churn_rate_pct", ascending=True)
ax = plot_data.plot(
    x="PaymentMethod", y="churn_rate_pct", kind="barh", figsize=(8,4),
    legend=False, title="Taxa de Churn por PaymentMethod"
)
ax.set_xlabel("Churn (%)")
ax.set_ylabel("PaymentMethod")
plt.tight_layout()
plt.show()


## TechSupport — churn por categoria


In [ ]:
result = churn_by_category(df, "TechSupport")
result


In [ ]:
plot_data = result.sort_values("churn_rate_pct", ascending=True)
ax = plot_data.plot(
    x="TechSupport", y="churn_rate_pct", kind="barh", figsize=(8,4),
    legend=False, title="Taxa de Churn por TechSupport"
)
ax.set_xlabel("Churn (%)")
ax.set_ylabel("TechSupport")
plt.tight_layout()
plt.show()


## OnlineSecurity — churn por categoria


In [ ]:
result = churn_by_category(df, "OnlineSecurity")
result


In [ ]:
plot_data = result.sort_values("churn_rate_pct", ascending=True)
ax = plot_data.plot(
    x="OnlineSecurity", y="churn_rate_pct", kind="barh", figsize=(8,4),
    legend=False, title="Taxa de Churn por OnlineSecurity"
)
ax.set_xlabel("Churn (%)")
ax.set_ylabel("OnlineSecurity")
plt.tight_layout()
plt.show()


## PaperlessBilling — churn por categoria


In [ ]:
result = churn_by_category(df, "PaperlessBilling")
result


In [ ]:
plot_data = result.sort_values("churn_rate_pct", ascending=True)
ax = plot_data.plot(
    x="PaperlessBilling", y="churn_rate_pct", kind="barh", figsize=(8,4),
    legend=False, title="Taxa de Churn por PaperlessBilling"
)
ax.set_xlabel("Churn (%)")
ax.set_ylabel("PaperlessBilling")
plt.tight_layout()
plt.show()


## 16. Varredura de todas as variáveis categóricas


In [ ]:
for col in categorical_cols:
    print("="*80)
    print(col)
    display(churn_by_category(df, col))


## 17. Tenure por faixas


In [ ]:
eda_df = df.copy()
eda_df["tenure_group"] = pd.cut(
    eda_df["tenure"],
    bins=[-1,6,12,24,36,48,60,72],
    labels=["0-6","7-12","13-24","25-36","37-48","49-60","61-72"]
)
tenure_analysis = churn_by_category(eda_df, "tenure_group")
tenure_analysis


In [ ]:
tenure_plot = tenure_analysis.set_index("tenure_group").reindex(
    ["0-6","7-12","13-24","25-36","37-48","49-60","61-72"]
)
ax = tenure_plot["churn_rate_pct"].plot(
    kind="bar", figsize=(9,5), title="Taxa de Churn por Tempo de Relacionamento"
)
ax.set_xlabel("Faixa de tenure")
ax.set_ylabel("Churn (%)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 18. MonthlyCharges por faixas


In [ ]:
eda_df["monthly_charge_group"] = pd.qcut(eda_df["MonthlyCharges"], q=5, duplicates="drop")
monthly_charge_analysis = churn_by_category(eda_df, "monthly_charge_group")
monthly_charge_analysis


In [ ]:
plot_data = monthly_charge_analysis.sort_values("monthly_charge_group")
ax = plot_data.plot(
    x="monthly_charge_group", y="churn_rate_pct", kind="bar", figsize=(10,5),
    legend=False, title="Taxa de Churn por Faixa de MonthlyCharges"
)
ax.set_xlabel("Faixa de MonthlyCharges")
ax.set_ylabel("Churn (%)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


## 19. TotalCharges por faixas


In [ ]:
eda_df["total_charge_group"] = pd.qcut(eda_df["TotalCharges"], q=5, duplicates="drop")
churn_by_category(eda_df, "total_charge_group")


## 20. Relação entre tenure e MonthlyCharges


In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
for churn_value, label in [(0,"Não Churn"),(1,"Churn")]:
    subset = df[df["Churn"] == churn_value]
    ax.scatter(subset["tenure"], subset["MonthlyCharges"], alpha=0.35, s=18, label=label)
ax.set_title("Tenure x MonthlyCharges por Churn")
ax.set_xlabel("tenure")
ax.set_ylabel("MonthlyCharges")
ax.legend()
plt.tight_layout()
plt.show()


## 21. Correlação entre variáveis numéricas


In [ ]:
corr_cols = numeric_cols + ["Churn"]
corr = df[corr_cols].corr()
corr


In [ ]:
fig, ax = plt.subplots(figsize=(7,5))
im = ax.imshow(corr.values)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticklabels(corr.columns)
for i in range(len(corr.index)):
    for j in range(len(corr.columns)):
        ax.text(j, i, f"{corr.iloc[i,j]:.2f}", ha="center", va="center")
ax.set_title("Correlação entre variáveis numéricas")
fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()


## 22. Associação categórica com Churn — Cramér's V

Útil para ordenar a força de associação das categorias com o target. Associação não implica causalidade.


In [ ]:
from scipy.stats import chi2_contingency

def cramers_v(data: pd.DataFrame, x: str, y: str = "Churn") -> float:
    contingency = pd.crosstab(data[x], data[y])
    chi2 = chi2_contingency(contingency)[0]
    n = contingency.values.sum()
    phi2 = chi2 / n
    r, k = contingency.shape
    phi2_corr = max(0, phi2 - ((k-1)*(r-1))/(n-1))
    r_corr = r - ((r-1)**2)/(n-1)
    k_corr = k - ((k-1)**2)/(n-1)
    denominator = min(k_corr-1, r_corr-1)
    return np.sqrt(phi2_corr / denominator) if denominator > 0 else np.nan

categorical_association = pd.DataFrame([
    {"feature": col, "cramers_v": cramers_v(df, col)}
    for col in categorical_cols
]).sort_values("cramers_v", ascending=False).reset_index(drop=True)

categorical_association


## 23. Combinações de variáveis — segmentos de risco


In [ ]:
contract_internet = (
    df.groupby(["Contract","InternetService"], observed=False)
    .agg(clientes=("Churn","size"), churn_rate=("Churn","mean"))
    .reset_index()
)
contract_internet["churn_rate_pct"] = contract_internet["churn_rate"] * 100
contract_internet.sort_values(["churn_rate","clientes"], ascending=[False,False])


In [ ]:
contract_support = (
    df.groupby(["Contract","TechSupport"], observed=False)
    .agg(clientes=("Churn","size"), churn_rate=("Churn","mean"))
    .reset_index()
)
contract_support["churn_rate_pct"] = contract_support["churn_rate"] * 100
contract_support.sort_values(["churn_rate","clientes"], ascending=[False,False])


## 24. Possíveis features derivadas

As features abaixo são exploratórias. Só devem entrar no modelo se melhorarem a validação cruzada.


In [ ]:
feature_df = df.copy()
feature_df["avg_charge_per_month"] = feature_df["TotalCharges"] / feature_df["tenure"].replace(0,1)

service_cols = [
    "PhoneService","MultipleLines","OnlineSecurity","OnlineBackup",
    "DeviceProtection","TechSupport","StreamingTV","StreamingMovies"
]
feature_df["num_active_services"] = feature_df[service_cols].apply(
    lambda s: s.eq("Yes")
).sum(axis=1)

feature_df[[
    "tenure","MonthlyCharges","TotalCharges",
    "avg_charge_per_month","num_active_services","Churn"
]].describe().T


In [ ]:
feature_df.groupby("Churn")[["avg_charge_per_month","num_active_services"]].agg(["mean","median"]).round(2)


## 25. Ranking descritivo de categorias com maior churn


In [ ]:
segment_rows = []
for col in categorical_cols:
    temp = churn_by_category(df, col).copy()
    temp["feature"] = col
    temp = temp.rename(columns={col: "category"})
    segment_rows.append(temp[["feature","category","clientes","churns","churn_rate_pct"]])

segment_ranking = pd.concat(segment_rows, ignore_index=True).sort_values(
    ["churn_rate_pct","clientes"], ascending=[False,False]
).reset_index(drop=True)
segment_ranking.head(30)


## 26. Resumo automático da EDA


In [ ]:
print("="*70)
print("RESUMO EDA")
print("="*70)
print(f"Clientes: {len(df):,}")
print(f"Colunas: {df.shape[1]}")
print(f"Taxa de churn: {df['Churn'].mean():.2%}")
print(f"Duplicados: {df.duplicated().sum():,}")
print(f"Nulos: {df.isna().sum().sum():,}")

print("
Médias numéricas por classe:")
display(df.groupby("Churn")[numeric_cols].mean().round(2))

print("
Top associações categóricas por Cramér's V:")
display(categorical_association.head(10))

print("
Categorias com maior churn:")
display(segment_ranking.head(15))


# 27. Conclusões da EDA

Preencha esta seção **após executar o notebook**, sempre usando os números reais obtidos.

### Qualidade dos dados
- A base contém **7.043 clientes** e **21 colunas**.
- `customerID` identifica unicamente cada cliente e deve ser excluído da modelagem.
- Registrar aqui duplicidades, nulos e qualquer inconsistência observada.

### Distribuição do target
- Churn: aproximadamente **35,2%**.
- Não churn: aproximadamente **64,8%**.
- O desbalanceamento é moderado e não justifica oversampling automático.

### Principais padrões
1. **Contrato:** identificar qual modalidade apresenta maior churn.
2. **Tempo de relacionamento:** avaliar se clientes recentes concentram mais churn.
3. **Cobrança mensal:** verificar se faixas mais altas apresentam maior churn.
4. **Internet e serviços:** comparar InternetService, TechSupport e OnlineSecurity.
5. **Pagamento:** identificar métodos com maior taxa de churn.

### Implicações para modelagem
- Remover `customerID`.
- Tratar `SeniorCitizen` como categórica/binária.
- Comparar Logistic Regression, Random Forest e XGBoost.
- Comparar modelos com e sem balanceamento.
- Avaliar ROC-AUC, PR-AUC, Precision, Recall e F1.
- Usar Lift@K para aproximar a avaliação do uso real em retenção.
- Testar features derivadas apenas se melhorarem CV.

### Implicações de negócio
O score deve priorizar clientes para retenção. Porém, risco de churn não equivale a resposta incremental a uma oferta. Uma evolução natural seria validar a ação com tratamento × controle ou uplift modeling.


# 28. Checklist antes da modelagem

- [ ] Base carregada corretamente
- [ ] `customerID` único
- [ ] Duplicidades avaliadas
- [ ] Missing values avaliados
- [ ] Target documentado
- [ ] Numéricas analisadas
- [ ] Categóricas analisadas
- [ ] Segmentos de maior churn identificados
- [ ] Possíveis features derivadas registradas
- [ ] Insights escritos com números reais
- [ ] Nenhuma conclusão causal feita apenas a partir da EDA
